In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJECT_ROOT / "data"

# Game & Player Context
- SEASON_ID: A unique code for the NBA season. For example, "22019" typically denotes the regular season (2) of the 2019-2020 season.

- Player_ID: The unique numerical identifier the NBA database assigns to this specific player.

- Game_ID: The unique numerical identifier for this specific game.

- GAME_DATE: The date the game was played.

- MATCHUP: The teams playing. "vs." means a home game (e.g., NOP vs. SAS means New Orleans hosted San Antonio). "@" means an away game (e.g., NOP @ SAC means New Orleans traveled to play Sacramento).

- WL: The game's outcome for the player's team (Win or Loss).

- MIN: Minutes the player spent on the court during that game.

- PLAYER_NAME: The player's full name (Zion Williamson in this preview).

# Shooting Statistics
- FGM (Field Goals Made): Total number of baskets made (includes both 2-pointers and 3-pointers, but not free throws).

- FGA (Field Goals Attempted): Total number of shots taken.

- FG_PCT (Field Goal Percentage): The percentage of shots made (FGM / FGA).

- FG3M (3-Point Field Goals Made): Number of 3-point shots made.

- FG3A (3-Point Field Goals Attempted): Number of 3-point shots taken.

- FG3_PCT (3-Point Percentage): The percentage of 3-point shots made (FG3M / FG3A).

- FTM (Free Throws Made): Number of successful free throws.

- FTA (Free Throws Attempted): Number of free throws taken.

- FT_PCT (Free Throw Percentage): The percentage of free throws made (FTM / FTA).

# Rebounding & Playmaking
- OREB (Offensive Rebounds): Rebounds grabbed while the player's team is on offense (giving them another chance to score).

- DREB (Defensive Rebounds): Rebounds grabbed while the player's team is on defense.

- REB (Total Rebounds): The sum of Offensive and Defensive rebounds (OREB + DREB).

- AST (Assists): Passes that lead directly to a teammate scoring a basket.

# Defense & Mistakes
- STL (Steals): Number of times the player legally takes the ball away from an opponent.

- BLK (Blocks): Number of times the player successfully deflects an opponent's shot attempt.

- TOV (Turnovers): Number of times the player loses possession of the ball to the other team (via a bad pass, traveling, getting stolen from, etc.).

- PF (Personal Fouls): Number of fouls committed by the player in that game.

# Overall Impact
- PTS (Points): Total points scored by the player in that game.

- PLUS_MINUS (+/-): A metric showing the team's point differential while that specific player was on the court. A "+8" means the team scored 8 more points than the opponent while Zion was playing. A "-21" means they were outscored by 21 points while he was on the floor.

- VIDEO_AVAILABLE: A backend database flag (1 = Yes, 0 = No) indicating if video clips of this game are linked in the NBA's API.


# Proposed Aggregated Features

- Game Score (GmSc) = PTS + 0.4 * FGM - 0.7 * FGA - 0.4 * (FTA - FTM) + 0.7 * OREB + 0.3 * DREB + STL + 0.7 * AST + 0.7 * BLK - 0.4 * PF - TOV
  Created by John Hollinger, this is essentially a "single-game PER" (Player Efficiency Rating). It gives a rough, all-in-one number indicating how productive a player was in a specific game. A score of 10 is average, and 40 is an all-time great performance.

- Value Point System (VPS) = (PTS + REB + 2 * AST + 2 * (STL + BLK)) / (2 * (FGA - FGM) + (FTA - FTM) + 2 * PF + 2 * TOV)
  VPS is based on a formula that assesses performance based upon several offensive and defensive stats. The higher the number, the better. A VPS of 1 is about average.

- True Shooting Percentage (TS%) = PTS / (2 * (FGA + 0.44 * FTA))
  True Shooting Percentage measures a player's efficiency at shooting the ball across all types of shots.

- NOTE: If we look at seasonal total, we should aggregate to the same game length, example: (Total Season Stat / Total Season Minutes) * 36

In [12]:
career_total = pd.read_csv(DATA_DIR / "career_totals_targets.csv")

# ==========================================
# 1. True Shooting Percentage (TS_PCT)
# ==========================================
# Formula: PTS / (2 * (FGA + 0.44 * FTA))
ts_denominator = 2 * (career_total['FGA'] + (0.44 * career_total['FTA']))

# Use np.where to avoid Division By Zero errors for players with 0 shot attempts
career_total['TS_PCT'] = np.where(
    ts_denominator == 0, 
    0, 
    career_total['PTS'] / ts_denominator
)

# ==========================================
# 2. Value Point System (VPS)
# ==========================================
# Numerator: PTS + REB + 2*AST + 2*(STL + BLK)
vps_num = (career_total['PTS'] + 
           career_total['REB'] + 
           (2 * career_total['AST']) + 
           (2 * (career_total['STL'] + career_total['BLK'])))

# Denominator: 2*(Missed FGs) + (Missed FTs) + 2*PF + 2*TOV
vps_den = ((2 * (career_total['FGA'] - career_total['FGM'])) + 
           (career_total['FTA'] - career_total['FTM']) + 
           (2 * career_total['PF']) + 
           (2 * career_total['TOV']))

career_total['VPS'] = np.where(
    vps_den == 0, 
    0, 
    vps_num / vps_den
)

# ==========================================
# 3. Game Score (GmSc) 
# ==========================================
# First, calculate raw career Total Game Score
raw_gmsc = (career_total['PTS'] + 
            (0.4 * career_total['FGM']) - 
            (0.7 * career_total['FGA']) - 
            (0.4 * (career_total['FTA'] - career_total['FTM'])) + 
            (0.7 * career_total['OREB']) + 
            (0.3 * career_total['DREB']) + 
            career_total['STL'] + 
            (0.7 * career_total['AST']) + 
            (0.7 * career_total['BLK']) - 
            (0.4 * career_total['PF']) - 
            career_total['TOV'])

# Normalize to "Per 48 Minutes" for accurate player-to-player evaluation
career_total['GmSc_Per48'] = np.where(
    career_total['MIN'] == 0,
    0,
    raw_gmsc * (48 / career_total['MIN'])
)

# Preview the calculated metrics
cols_to_show = ['PLAYER_NAME', 'SEASON_ID', 'MIN', 'TS_PCT', 'VPS', 'GmSc_Per48']

display(career_total[cols_to_show].head(20))

career_total.to_csv(DATA_DIR / "career_totals_targets_with_performance_analysis.csv", index=False)

,PLAYER_NAME,SEASON_ID,MIN,TS_PCT,VPS,GmSc_Per48
0,Zion Williamson,2019-20,668,0.615988,1.484099,28.634731
1,Zion Williamson,2020-21,2026,0.648548,1.740127,31.541165
2,Zion Williamson,2022-23,956,0.652114,1.718383,30.446862
3,Zion Williamson,2023-24,2207,0.610435,1.652150,28.193203
4,Zion Williamson,2024-25,857,0.600124,1.613610,33.773629
5,Zion Williamson,2025-26,1542,0.636595,1.693322,28.681712
6,Ja Morant,2019-20,2074,0.556167,1.509655,21.630087
7,Ja Morant,2020-21,2053,0.537174,1.455908,21.479591
8,Ja Morant,2021-22,1889,0.575169,1.525579,29.999365
9,Ja Morant,2022-23,1948,0.556998,1.526316,30.317864


In [9]:
career_total.columns

Index(['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A',
       'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL',
       'BLK', 'TOV', 'PF', 'PTS', 'PLAYER_NAME', 'TS_PCT', 'VPS',
       'GmSc_Per48'],
      dtype='object')